In [ ]:
# CELL 0 — Mount Drive & Unzip
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

# ⚠️ Change this to the actual path of your zip in Google Drive
zip_path     = '/content/drive/MyDrive/fruitinAmazon.zip'
extract_path = '/content/fruitinAmazon'

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

# Show folder structure so you can confirm train_dir below
for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, '').count(os.sep)
    if level < 3:
        print('  ' * level + os.path.basename(root) + '/')

In [ ]:
# CELL 1 — Imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import random
from PIL import Image
from sklearn.metrics import classification_report

print('TF version:', tf.__version__)

In [ ]:
# CELL 2 — Set path (update if your structure is different after Cell 0 output)
train_dir = '/content/fruitinAmazon/train'
# try '/content/fruitinAmazon/fruitinAmazon/train' if the above fails

class_dirs = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
print('Classes:', class_dirs)

In [ ]:
# CELL 3 — Task 1a: Visualise one image per class
sample_images, sample_labels = [], []
for cls in class_dirs:
    cls_path = os.path.join(train_dir, cls)
    imgs = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if imgs:
        sample_images.append(os.path.join(cls_path, random.choice(imgs)))
        sample_labels.append(cls)

n = len(sample_images)
cols = (n + 1) // 2
fig, axes = plt.subplots(2, cols, figsize=(cols * 3, 6))
axes = axes.flatten()
for i, (p, l) in enumerate(zip(sample_images, sample_labels)):
    axes[i].imshow(Image.open(p))
    axes[i].set_title(l, fontsize=9)
    axes[i].axis('off')
for j in range(i+1, len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 4 — Task 1b: Check for corrupted images
corrupted = []
for cls in class_dirs:
    cls_path = os.path.join(train_dir, cls)
    for fname in os.listdir(cls_path):
        fpath = os.path.join(cls_path, fname)
        try:
            img = Image.open(fpath)
            img.verify()
        except (IOError, SyntaxError):
            print(f'Removed corrupted image: {fpath}')
            os.remove(fpath)
            corrupted.append(fpath)

if not corrupted:
    print('No corrupted images found.')
else:
    print(f'Total removed: {len(corrupted)}')

In [ ]:
# CELL 5 — Task 2: Load & preprocess data
IMG_HEIGHT = 128
IMG_WIDTH  = 128
BATCH_SIZE = 16
VAL_SPLIT  = 0.2
SEED       = 123

rescale = tf.keras.layers.Rescaling(1.0 / 255)

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir, labels='inferred', label_mode='int',
    image_size=(IMG_HEIGHT, IMG_WIDTH), interpolation='nearest',
    batch_size=BATCH_SIZE, shuffle=True,
    validation_split=VAL_SPLIT, subset='training', seed=SEED
)
train_ds = train_ds.map(lambda x, y: (rescale(x), y))

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir, labels='inferred', label_mode='int',
    image_size=(IMG_HEIGHT, IMG_WIDTH), interpolation='nearest',
    batch_size=BATCH_SIZE, shuffle=False,
    validation_split=VAL_SPLIT, subset='validation', seed=SEED
)
val_ds = val_ds.map(lambda x, y: (rescale(x), y))

class_names = train_ds.class_names
num_classes = len(class_names)
print('Classes:', class_names)
print('Num classes:', num_classes)

In [ ]:
# CELL 6 — Task 3: Build CNN
model = keras.Sequential([
    layers.Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.MaxPooling2D((2,2), strides=2),
    layers.Conv2D(32, (3,3), padding='same', activation='relu'),
    layers.MaxPooling2D((2,2), strides=2),
    layers.Flatten(),
    layers.Dense(64,  activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])
model.summary()

In [ ]:
# CELL 7 — Task 4: Compile & Train
model.compile(
    optimizer = 'adam',
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy']
)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
)
checkpoint = keras.callbacks.ModelCheckpoint(
    'best_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1
)

history = model.fit(
    train_ds,
    epochs=250,
    validation_data=val_ds,
    callbacks=[early_stop, checkpoint]
)

In [ ]:
# CELL 8 — Task 5: Evaluate & plot curves
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f'Validation Loss:     {val_loss:.4f}')
print(f'Validation Accuracy: {val_acc:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# CELL 9 — Task 6: Save & reload model
model.save('cnn_fruit_model.h5')
print('Model saved.')

loaded_model = keras.models.load_model('cnn_fruit_model.h5')
loss2, acc2 = loaded_model.evaluate(val_ds, verbose=0)
print(f'Reloaded model — Loss: {loss2:.4f}  Accuracy: {acc2:.4f}')

# Uncomment to download for submission:
# from google.colab import files
# files.download('cnn_fruit_model.h5')

In [ ]:
# CELL 10 — Task 7: Predictions & Classification Report
all_true, all_preds = [], []
for images, labels in val_ds:
    preds = loaded_model.predict(images, verbose=0)
    all_preds.extend(np.argmax(preds, axis=1))
    all_true.extend(labels.numpy())

print(classification_report(all_true, all_preds, target_names=class_names))

# Visualise 5 sample predictions
imgs_batch, lbls_batch = next(iter(val_ds))
preds5 = np.argmax(loaded_model.predict(imgs_batch[:5], verbose=0), axis=1)
true5  = lbls_batch[:5].numpy()

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for i in range(5):
    axes[i].imshow(imgs_batch[i].numpy())
    axes[i].set_title(
        f'P: {class_names[preds5[i]]}\nT: {class_names[true5[i]]}',
        color='green' if preds5[i]==true5[i] else 'red', fontsize=8
    )
    axes[i].axis('off')
plt.tight_layout()
plt.show()